# Gamma Blast — Backtest Evaluation

Evaluates the long-straddle Gamma Blast strategy (`/straddle` signal) backtested by `backtest_gamma_blast.py`.

**Rules:** after 12:30 IST, on a day where NIFTY spot moved <0.60% till 12:30, buy the ATM straddle when a combined-premium candle closes above EMA21 and the next candle breaks its high. SL = setup-candle low (capped 25 pts), exit EOD.

This notebook reads `gamma_blast_trades.csv` (one row per trade, generated with `--dump`), so it runs anywhere — no options data needed. Regenerate the CSV with:
```
python backtest_gamma_blast.py --tf 5m --dump gamma_blast_trades.csv
```
PnL is in **premium points**; ₹ = points × lot (75). Positive = long straddle profit.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': .25,
                     'axes.spines.top': False, 'axes.spines.right': False})
LOT = 75

tr = pd.read_csv('gamma_blast_trades.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
tr['cum'] = tr['pnl'].cumsum()
tr['win'] = tr['pnl'] > 0
tr['year'] = tr['date'].dt.year
print(f"{len(tr)} trades  |  {tr['date'].min().date()} → {tr['date'].max().date()}")
tr.head()

## Headline summary

In [ ]:
def summary(p):
    p = np.asarray(p, float); w = p[p > 0]; l = p[p <= 0]
    return dict(trades=len(p), win_pct=round(len(w)/len(p)*100, 1),
                avg_pts=round(p.mean(), 2), total_pts=round(p.sum(), 0),
                median=round(np.median(p), 2), avg_win=round(w.mean(), 1) if len(w) else 0,
                avg_loss=round(l.mean(), 1) if len(l) else 0,
                best=round(p.max(), 1), worst=round(p.min(), 1),
                avg_rupees=round(p.mean()*LOT, 0), payoff=round(abs(w.mean()/l.mean()), 2) if len(l) and len(w) else np.nan)

s = summary(tr['pnl'])
for k, v in s.items():
    print(f"  {k:<11}: {v}")
print(f"\n  expectancy = {s['avg_pts']} pts/trade (₹{s['avg_rupees']:.0f}/lot) — asymmetric: {s['win_pct']}% win but payoff {s['payoff']}:1")

## Equity curve (cumulative points)

In [ ]:
fig, ax = plt.subplots()
ax.plot(tr['date'], tr['cum'], color='#22c55e', lw=1.5)
ax.fill_between(tr['date'], tr['cum'], 0, color='#22c55e', alpha=.12)
# max drawdown
peak = tr['cum'].cummax(); dd = tr['cum'] - peak
ax.set_title(f"Equity curve  |  total {tr['cum'].iloc[-1]:.0f} pts  |  max drawdown {dd.min():.0f} pts")
ax.set_ylabel('cumulative pts'); plt.show()
print(f"max drawdown: {dd.min():.0f} pts (₹{dd.min()*LOT:,.0f})  |  final: {tr['cum'].iloc[-1]:.0f} pts")

## PnL distribution — the asymmetry (many small stops, few big winners)

In [ ]:
fig, ax = plt.subplots()
ax.hist(tr['pnl'].clip(-30, 120), bins=60, color='#60a5fa', edgecolor='#0d1320')
ax.axvline(0, color='#e6edf6', lw=1); ax.axvline(tr['pnl'].mean(), color='#f59e0b', ls='--', label=f"mean {tr['pnl'].mean():.1f}")
ax.set_title('Per-trade PnL (points, clipped for display)'); ax.set_xlabel('points'); ax.legend(); plt.show()
print(f"outcomes: {tr['outcome'].value_counts().to_dict()}")

## By days-to-expiry (is it an expiry-day edge?)

In [ ]:
g = tr.groupby('dte').agg(n=('pnl', 'size'), avg=('pnl', 'mean'),
                          total=('pnl', 'sum'), win=('win', 'mean')).reset_index()
g = g[g['n'] >= 5]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#22c55e' if v > 0 else '#f87171' for v in g['avg']]
a1.bar(g['dte'], g['avg'], color=colors); a1.set_title('avg PnL by DTE (pts)'); a1.set_xlabel('days to expiry')
a2.bar(g['dte'], g['win']*100, color='#a78bfa'); a2.set_title('win % by DTE'); a2.set_xlabel('days to expiry')
plt.tight_layout(); plt.show()
g.round(2)

## Yearly performance

In [ ]:
y = tr.groupby('year').agg(n=('pnl', 'size'), total=('pnl', 'sum'),
                           avg=('pnl', 'mean'), win=('win', 'mean')).reset_index()
fig, ax = plt.subplots()
ax.bar(y['year'], y['total'], color=['#22c55e' if v > 0 else '#f87171' for v in y['total']])
ax.set_title('Total PnL by year (points)'); ax.set_ylabel('pts'); plt.show()
y.assign(win=lambda d: (d['win']*100).round(1)).round(2)

## Cost sensitivity — the edge is thin
A long straddle is 4 option legs (2 in, 2 out). Net edge/trade = gross − round-trip cost (points).

In [ ]:
gross = tr['pnl'].mean(); costs = np.arange(0, 6.01, 0.5)
net = gross - costs
fig, ax = plt.subplots()
ax.plot(costs, net, marker='o', color='#22c55e')
ax.axhline(0, color='#f87171', ls='--'); ax.fill_between(costs, net, 0, where=net>=0, color='#22c55e', alpha=.12)
be = gross  # breakeven cost = gross edge
ax.set_title(f'Net edge vs round-trip cost  |  breakeven ≈ {be:.2f} pts'); ax.set_xlabel('round-trip cost (pts, 4 legs)'); ax.set_ylabel('net pts/trade'); plt.show()
print(f"breakeven cost ≈ {be:.2f} pts round-trip. Only viable if execution keeps 4-leg cost under that.")

## Does the EMA21 breakout beat just buying at 12:30? (control)

In [ ]:
if 'control_1230' in tr and tr['control_1230'].notna().any():
    c = tr['control_1230'].dropna()
    sig_s = summary(tr.loc[c.index, 'pnl']); ctl_s = summary(c)
    print(f"SIGNAL (breakout) : avg {sig_s['avg_pts']:+.2f} pts  win {sig_s['win_pct']}%  total {sig_s['total_pts']:.0f}")
    print(f"CONTROL (12:30 buy): avg {ctl_s['avg_pts']:+.2f} pts  win {ctl_s['win_pct']}%  total {ctl_s['total_pts']:.0f}")
    print(f"\nbreakout edge over 12:30-buy: {sig_s['avg_pts']-ctl_s['avg_pts']:+.2f} pts/trade")
    fig, ax = plt.subplots()
    ax.plot(tr['date'], tr['pnl'].cumsum(), label='signal (breakout)', color='#22c55e')
    ax.plot(tr['date'], tr['control_1230'].cumsum(), label='control (12:30 buy)', color='#8a98b2')
    ax.set_title('Cumulative PnL: signal vs 12:30 control'); ax.legend(); ax.set_ylabel('pts'); plt.show()
else:
    print('control_1230 column not present in this dump')

## Biggest winners & losers

In [ ]:
cols = ['date', 'dte', 'atm', 'entry_time', 'spot_move', 'level', 'sl', 'pnl', 'rupees', 'outcome']
print('TOP 8 winners:'); display(tr.nlargest(8, 'pnl')[cols])
print('TOP 8 losers:');  display(tr.nsmallest(8, 'pnl')[cols])